1. Imports

In [45]:
from sentence_transformers import SentenceTransformer, util
from transformers import BertTokenizer, BertModel
import csv
import pandas as pd
import torch
import torch.nn.functional as F

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2") #S-BERT Model

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9275.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [98]:
clozeDf = pd.read_csv("D://Python//MAForumProject//cloze_winter_2018_val.csv", sep = ",")

clozeDf.head(5)

,InputStoryid,InputSentence1,InputSentence2,InputSentence3,InputSentence4,RandomFifthSentenceQuiz1,RandomFifthSentenceQuiz2,AnswerRightEnding
0,138d5bfb-05cc-41e3-bf2c-fa85ebad14e2,Rick grew up in a troubled household.,"He never found good support in family, and tur...",It wasn't long before Rick got shot in a robbery.,The incident caused him to turn a new leaf.,He is happy now.,He joined a gang.,1
1,bff9f820-9605-4875-b9af-fe6f14d04256,Laverne needs to prepare something for her fri...,She decides to bake a batch of brownies.,She chooses a recipe and follows it closely.,Laverne tests one of the brownies to make sure...,The brownies are so delicious Laverne eats two...,Laverne doesn't go to her friend's party.,1
2,e8f628d5-9f97-40ed-8611-fc0e774673c4,Sarah had been dreaming of visiting Europe for...,She had finally saved enough for the trip.,She landed in Spain and traveled east across t...,She didn't like how different everything was.,Sarah then decided to move to Europe.,Sarah decided that she preferred her home over...,2
3,f5226bfe-9f26-4377-b05f-3d9568dbdec1,Gina was worried the cookie dough in the tube ...,She was very happy to find she was wrong.,The cookies from the tube were as good as from...,Gina intended to only eat 2 cookies and save t...,Gina liked the cookies so much she ate them al...,Gina gave the cookies away at her church.,1
4,69ac9b05-b956-402f-9fff-1f926ef9176b,It was my final performance in marching band.,I was playing the snare drum in the band.,We played Thriller and Radar Love.,The performance was flawless.,I was very proud of my performance.,I was very ashamed of my performance.,1


2. Sentence by Sentence S-BERT Example

In [ ]:
sentences = ["I went to the beach.", "The beach was filled with jellyfish.", "I was sad that I couldn't swim.",
     "I had an idea.", "I decided to swim in the pool instead."]


In [10]:
embeddings = model.encode(sentences, normalize_embeddings=True)
print(embeddings.shape)

(5, 384)


In [5]:
similarities = util.pytorch_cos_sim(embeddings, embeddings)
for j in range(len(similarities)):
    print("{}. Sentence = {}\nCosine Similarities of Sentence = {}\n".format(j, sentences[j], similarities[j]))
# tensor([[1.0000, 0.6660, 0.1046],
#         [0.6660, 1.0000, 0.1411],
#         [0.1046, 0.1411, 1.0000]])
print(len(similarities))

0. Sentence = I went to the beach.
Cosine Similarities of Sentence = tensor([1.0000, 0.4685, 0.4733, 0.2741, 0.5814])

1. Sentence = The beach was filled with jellyfish.
Cosine Similarities of Sentence = tensor([0.4685, 1.0000, 0.2952, 0.0968, 0.2877])

2. Sentence = I was sad that I couldn't swim.
Cosine Similarities of Sentence = tensor([0.4733, 0.2952, 1.0000, 0.1961, 0.6703])

3. Sentence = I had an idea.
Cosine Similarities of Sentence = tensor([0.2741, 0.0968, 0.1961, 1.0000, 0.2635])

4. Sentence = I decided to swim in the pool instead.
Cosine Similarities of Sentence = tensor([0.5814, 0.2877, 0.6703, 0.2635, 1.0000])

5


3. S-BERT Sentence by Sentence Metric Formulation

In [16]:
clozeCosSims = {}
clozeStoryRefs = {}
clozeStoryAns = {}

i = 0

for index, row in clozeDf.iterrows():
        clozeCosSims[row["InputStoryid"]] = []
        clozeStoryRefs[row["InputStoryid"]] = []
        
        s4Embedding = model.encode(str(row["InputSentence4"]), normalize_embeddings=True)
        
        e1Embedding = model.encode(str(row["RandomFifthSentenceQuiz1"]), normalize_embeddings=True)

        firstSimilarities = util.pytorch_cos_sim(s4Embedding, e1Embedding)

        clozeCosSims[row["InputStoryid"]].append(firstSimilarities)
        clozeStoryRefs[row["InputStoryid"]].append([s4Embedding, e1Embedding])

        e2Embedding = model.encode(str(row["RandomFifthSentenceQuiz2"]), normalize_embeddings=True)

        secondSimilarities = util.pytorch_cos_sim(s4Embedding, e2Embedding)

        clozeCosSims[row["InputStoryid"]].append(secondSimilarities)
        clozeStoryRefs[row["InputStoryid"]].append([s4Embedding, e2Embedding])

        clozeStoryAns[str(row["InputStoryid"])] = int(row["AnswerRightEnding"])

        if i % 100 == 0:
                print("{} Stories Processed".format(i))

        i = i + 1



0 Stories Processed
100 Stories Processed
200 Stories Processed
300 Stories Processed
400 Stories Processed
500 Stories Processed
600 Stories Processed
700 Stories Processed
800 Stories Processed
900 Stories Processed
1000 Stories Processed
1100 Stories Processed
1200 Stories Processed
1300 Stories Processed
1400 Stories Processed
1500 Stories Processed


In [27]:
key = clozeDf["InputStoryid"]
for entry in key:
    print(entry)
    #print(clozeStoryRefs[entry])
    print(float(clozeCosSims[entry][0][0][0]))
    print(float(clozeCosSims[entry][1][0][0]))
    print(clozeStoryAns[entry])
    #Consider the differences in cosine similarities between index 4 and 5 (sentences 4 and 5).
    break

138d5bfb-05cc-41e3-bf2c-fa85ebad14e2
0.29694482684135437
0.3367763161659241
1


In [29]:
key = clozeDf["InputStoryid"]

cosSimsS4E1 = []
cosSimsS4E2 = []
ansRightEnding = []

for entry in key:
    cosSimsS4E1.append(float(clozeCosSims[entry][0][0][0]))
    cosSimsS4E2.append(float(clozeCosSims[entry][1][0][0]))
    ansRightEnding.append(clozeStoryAns[entry])


In [34]:
with open("sbertSentBySentCosSims.csv", "w") as file:
    writer = csv.writer(file)
    writer.writerow(["storyID", "S4E1 Similarities", "S4E2 Similarities", "Correct Ending"])
    
    for i in range(len(ansRightEnding)):
        writer.writerow([key[i], cosSimsS4E1[i], cosSimsS4E2[i], ansRightEnding[i]])

In [86]:
sbertDf = pd.read_csv("./sbertSentBySentCosSims.csv")

In [87]:
sbertDf.head(5)

,storyID,S4E1 Similarities,S4E2 Similarities,Correct Ending
0,138d5bfb-05cc-41e3-bf2c-fa85ebad14e2,0.296945,0.336776,1
1,bff9f820-9605-4875-b9af-fe6f14d04256,0.842814,0.444635,1
2,e8f628d5-9f97-40ed-8611-fc0e774673c4,0.272204,0.368684,2
3,f5226bfe-9f26-4377-b05f-3d9568dbdec1,0.833238,0.709119,1
4,69ac9b05-b956-402f-9fff-1f926ef9176b,0.573658,0.486423,1


In [88]:
correctCount = 0
incorrectCount = 0

for row in sbertDf.iterrows():
    #print(row[1]['storyID'])

    if row[1]['Correct Ending'] == 1:
        if row[1]['S4E1 Similarities'] > row[1]['S4E2 Similarities']:
            correctCount += 1
        elif row[1]['S4E1 Similarities'] < row[1]['S4E2 Similarities']:
            incorrectCount += 1

    if row[1]['Correct Ending'] == 2:
        if row[1]['S4E1 Similarities'] > row[1]['S4E2 Similarities']:
            incorrectCount += 1
        elif row[1]['S4E1 Similarities'] < row[1]['S4E2 Similarities']:
            correctCount += 1

    

In [89]:
print("S-BERT Model Sentence by Sentence Cosine Similaritites, Correctly Identifies Matching Narrative Ending\n - Correct Narrative Ending Selected (n = {} times)\n - Incorrect Narrative Ending Selected (n = {} times).\nS-BERT Model Accuracy: {}".format(correctCount, incorrectCount, correctCount / (incorrectCount + correctCount)))

S-BERT Model Sentence by Sentence Cosine Similaritites, Correctly Identifies Matching Narrative Ending
 - Correct Narrative Ending Selected (n = 980 times)
 - Incorrect Narrative Ending Selected (n = 591 times).
S-BERT Model Accuracy: 0.6238064926798218


4. BERT Sentence by Sentence Metric Formulation

In [46]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10011.59it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [63]:
s4E1CosSims = 0
s4E2CosSims = 0

with open('bertSentBySentCosSims.csv', 'w', newline = '') as file:
    writer = csv.writer(file)

    writer.writerow(['StoryID','First Ending Sent', 'S4E1CosSims', 'Second Ending Sent', 'S4E2CosSims', 'Correct Answer'])

    #Init tokenizer

    for i, row in clozeDf.iterrows():
     

##GET BERT EMBEDDINGS FIRST ENDING
#Narrative sentences
        sentence4 = str(row['InputSentence4'])

        sentence5 = str(row['RandomFifthSentenceQuiz1'])

# Tokenize the sentences
  
        tokens4 = tokenizer.tokenize(sentence4)
         
        tokens5 = tokenizer.tokenize(sentence5)

        tokens = ['[CLS]'] + tokens4 + ['[SEP]'] + tokens5
      
        input_ids = tokenizer.convert_tokens_to_ids(tokens)

# Convert tokens to input IDs
  
        inputIds4 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens4)).unsqueeze(0)  

        inputIds5 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens5)).unsqueeze(0)  

# BERT embeddings
        with torch.no_grad():

                outputs4 = model(inputIds4)

                outputs5 = model(inputIds5)


        embeddings4 = F.normalize(outputs4.last_hidden_state[:, 0, :], p = 2, dim = 1)  
        
        embeddings5 = F.normalize(outputs5.last_hidden_state[:, 0, :], p = 2, dim = 1)  


        x4 = embeddings4[0].reshape(1, -1)
        x5 = embeddings5[0].reshape(1, -1)

#Computing Similarities

        s4E1CosSims = util.pytorch_cos_sim(x4, x5)

##SECOND ENDING

#Narrative sentences
        sentence4 = str(row['InputSentence4'])

        sentence5 = str(row['RandomFifthSentenceQuiz2'])

# Tokenize the sentences
      
        tokens4 = tokenizer.tokenize(sentence4)
         
        tokens5 = tokenizer.tokenize(sentence5)

        tokens = ['[CLS]'] + tokens4 + ['[SEP]'] + tokens5
      
        inputIds = tokenizer.convert_tokens_to_ids(tokens)

# Convert tokens to input IDs
        inputIds4 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens4)).unsqueeze(0) 

        inputIds5 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens5)).unsqueeze(0)  

# BERT embeddings
        with torch.no_grad():
                outputs4 = model(inputIds4)

                outputs5 = model(inputIds5)

        embeddings4 = F.normalize(outputs4.last_hidden_state[:, 0, :], p = 2, dim = 1)  
    
        embeddings5 = F.normalize(outputs5.last_hidden_state[:, 0, :], p = 2, dim = 1)  

#Clustering
        x4 = embeddings4[0].reshape(1, -1)
        x5 = embeddings5[0].reshape(1, -1)

        s4E2CosSims = util.pytorch_cos_sim(x4, x5)
        #print(sentence5, cluster_labels, distance_metric_two)


        writer.writerow([row['InputStoryid'], str(row['RandomFifthSentenceQuiz1']), float(s4E1CosSims[0][0]), str(row['RandomFifthSentenceQuiz2']), float(s4E2CosSims[0][0]), row['AnswerRightEnding']])
        
        if i % 100 == 0:
                print("{} Stories Processed".format(i))

        #cloze_bert_sims[row["InputStoryid"]].append(similarity_score)
        #cloze_bert_refs[row["InputStoryid"]].append(second_batch)

        #cloze_bert_ans[str(row["InputStoryid"])] = int(row["AnswerRightEnding"])



0 Stories Processed
100 Stories Processed
200 Stories Processed
300 Stories Processed
400 Stories Processed
500 Stories Processed
600 Stories Processed
700 Stories Processed
800 Stories Processed
900 Stories Processed
1000 Stories Processed
1100 Stories Processed
1200 Stories Processed
1300 Stories Processed
1400 Stories Processed
1500 Stories Processed


In [82]:
bertDf = pd.read_csv("bertSentBySentCosSims.csv")

In [83]:
bertDf.head(5)

,StoryID,First Ending Sent,S4E1CosSims,Second Ending Sent,S4E2CosSims,Correct Answer
0,138d5bfb-05cc-41e3-bf2c-fa85ebad14e2,He is happy now.,0.522551,He joined a gang.,0.472550,1
1,bff9f820-9605-4875-b9af-fe6f14d04256,The brownies are so delicious Laverne eats two...,0.685989,Laverne doesn't go to her friend's party.,0.693997,1
2,e8f628d5-9f97-40ed-8611-fc0e774673c4,Sarah then decided to move to Europe.,0.530687,Sarah decided that she preferred her home over...,0.691620,2
3,f5226bfe-9f26-4377-b05f-3d9568dbdec1,Gina liked the cookies so much she ate them al...,0.557086,Gina gave the cookies away at her church.,0.507014,1
4,69ac9b05-b956-402f-9fff-1f926ef9176b,I was very proud of my performance.,0.820335,I was very ashamed of my performance.,0.619275,1


In [84]:
correctCount = 0
incorrectCount = 0

for row in bertDf.iterrows():
    if row[1]['Correct Answer'] == 1:
        if row[1]['S4E1CosSims'] > row[1]['S4E2CosSims']:
            correctCount += 1
        elif row[1]['S4E1CosSims'] < row[1]['S4E2CosSims']:
            incorrectCount += 1

    if row[1]['Correct Answer'] == 2:
        if row[1]['S4E1CosSims'] > row[1]['S4E2CosSims']:
            incorrectCount += 1
        elif row[1]['S4E1CosSims'] < row[1]['S4E2CosSims']:
            correctCount += 1

In [85]:
print("BERT Model Sentence by Sentence Cosine Similaritites, Correctly Identifies Matching Narrative Ending\n - Correct Narrative Ending Selected (n = {} times)\n - Incorrect Narrative Ending Selected (n = {} times).\nBERT Model Accuracy: {}".format(correctCount, incorrectCount, correctCount / (incorrectCount + correctCount)))

BERT Model Sentence by Sentence Cosine Similaritites, Correctly Identifies Matching Narrative Ending
 - Correct Narrative Ending Selected (n = 778 times)
 - Incorrect Narrative Ending Selected (n = 793 times).
BERT Model Accuracy: 0.4952259707192871


5. S3BERT Sentence by Sentence Metric Formulation

- Note, original S3BERT implementation was developed and made available by (Opitz & Frank, 2022), cosine similarities were computed in the present study utilizing available pipeline at (https://github.com/flipz357/S3BERT) and adjustments as specified below. 

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import config
import prediction_helpers as ph
import torch
from importlib import reload


In [ ]:
#s3bert_infer.py is reconfigured to be within the function getFeatures(). This implementation was run in Google Colab.

def getFeatures(sentA, sentB):

  use_cuda = torch.cuda.is_available()
  device = torch.device("cuda" if use_cuda else "cpu")

# get decomposed model
  model = SentenceTransformer("./" + config.SBERT_SAVE_PATH + "/", device="cpu")

# example sentence pairs
  xsent = [sentA] 
  ysent = [sentB] 

# encode with s3bert
  xsent_encoded = model.encode(xsent, normalize_embeddings=True)
  ysent_encoded = model.encode(ysent, normalize_embeddings=True)

# get similarity scores of different features
  preds = ph.get_preds(xsent_encoded, ysent_encoded, biases=None, n=config.N, dim=config.FEATURE_DIM)

# print similarity scores of different features
  features = ["global"] + config.FEATURES[2:] + ["residual"]
  for i, x in enumerate(xsent):
    sims = preds[i]
    jl = {k:v for k,v in zip(features, sims)}
    jl["sent_a"] = x 
    jl["sent_b"] = ysent[i]
    #print(jl)

  return jl

In [ ]:
with open("s3bertSentbySentRaw.csv", "w") as f:
  writer = csv.writer(f)
  writer.writerow(["storyID", "narrativeSentenceFour", "firstEndSentence", "secondEndSentence", "cosSimsS4EndOne", "cosSimsS4EndTwo"])
  #for i in range(len(row_penultimate_sent)):
  for i in range(len(rowPenultimateSent)):
    reload(s3bert_infer)
    #reload(s3bert_infer)
    sentOne = rowPenultimateSent[i]
    sentTwo1 = rowFirstEnd[i]
    sentTwo2 = rowSecondEnd[i]

    #reload(s3bert_infer) Make sure s3bert_infer is configured in the .getFeatures() function.
    
    featOne = s3bert_infer.getFeatures(sentOne, sentTwo1)
    featTwo = s3bert_infer.getFeatures(sentOne, sentTwo2)
    writer.writerow([rowStoryID[i], rowPenultimateSent[i], rowFirstEnd[i], rowSecondEnd[i], featOne, featTwo])

    if i % 100 == 0:
      now = datetime.now()
      print("{} Stories Processed: {}".format(i, now))
      print(featOne, featTwo)

In [93]:
s3bertDf = pd.read_csv('./s3bertSentbySentRaw.csv')

s3bertDf.head(5)

,storyID,narrativeSentenceFour,firstEndSentence,secondEndSentence,cosSimsS4EndOne,cosSimsS4EndTwo
0,138d5bfb-05cc-41e3-bf2c-fa85ebad14e2,The incident caused him to turn a new leaf.,He is happy now.,He joined a gang.,"{'global': np.float64(0.2976914644241333), 'Co...","{'global': np.float64(0.39082208275794983), 'C..."
1,bff9f820-9605-4875-b9af-fe6f14d04256,Laverne tests one of the brownies to make sure...,The brownies are so delicious Laverne eats two...,Laverne doesn't go to her friend's party.,"{'global': np.float64(0.7701406478881836), 'Co...","{'global': np.float64(0.3981800377368927), 'Co..."
2,e8f628d5-9f97-40ed-8611-fc0e774673c4,She didn't like how different everything was.,Sarah then decided to move to Europe.,Sarah decided that she preferred her home over...,"{'global': np.float64(0.27726733684539795), 'C...","{'global': np.float64(0.4019138514995575), 'Co..."
3,f5226bfe-9f26-4377-b05f-3d9568dbdec1,Gina intended to only eat 2 cookies and save t...,Gina liked the cookies so much she ate them al...,Gina gave the cookies away at her church.,"{'global': np.float64(0.718230664730072), 'Con...","{'global': np.float64(0.5856587290763855), 'Co..."
4,69ac9b05-b956-402f-9fff-1f926ef9176b,The performance was flawless.,I was very proud of my performance.,I was very ashamed of my performance.,"{'global': np.float64(0.59031081199646), 'Conc...","{'global': np.float64(0.5324745178222656), 'Co..."


In [ ]:
#Extracting all global cosine similarity feature values and writing to csv file.
test_dict = ""
modif = 0
with open("s3bertSentBySentCosSims.csv", "w") as f:
  writer = csv.writer(f)
  writer.writerow(["storyID", "CosSimsS4E1", "CosSimsS4E2"])
  for index, row in s3bertDf.iterrows():
    global_cos_sims_one = row["cosSimsS4EndOne"]

    print(global_cos_sims_one)
    while global_cos_sims_one[22 + modif] != ")":
      modif = modif + 1
    #print(global_cos_sims_one[22: 22 + modif])
    global_cos_sims_one = float(global_cos_sims_one[22: 22 + modif])
    print(global_cos_sims_one)
    modif = 0

    global_cos_sims_two = row["cosSimsS4EndTwo"]

    print(global_cos_sims_two)
    while global_cos_sims_two[22 + modif] != ")":
      modif = modif + 1
    #print(global_cos_sims_two[22: 22 + modif])
    global_cos_sims_two = float(global_cos_sims_two[22: 22 + modif])
    print(global_cos_sims_two)
    modif = 0

    writer.writerow([row["storyID"], float(global_cos_sims_one), float(global_cos_sims_two)])

In [95]:
s3bertCosSimsDf = pd.read_csv("./s3bertSentBySentCosSims.csv")

s3bertCosSimsDf['CorrectAnswers'] = list(clozeDf["AnswerRightEnding"])

s3bertCosSimsDf.head(5)


,storyID,CosSimsS4E1,CosSimsS4E2,CorrectAnswers
0,138d5bfb-05cc-41e3-bf2c-fa85ebad14e2,0.297691,0.390822,1
1,bff9f820-9605-4875-b9af-fe6f14d04256,0.770141,0.398180,1
2,e8f628d5-9f97-40ed-8611-fc0e774673c4,0.277267,0.401914,2
3,f5226bfe-9f26-4377-b05f-3d9568dbdec1,0.718231,0.585659,1
4,69ac9b05-b956-402f-9fff-1f926ef9176b,0.590311,0.532475,1


In [96]:
correctCount = 0
incorrectCount = 0

for index, row in s3bertCosSimsDf.iterrows():
  if row["CorrectAnswers"] == 1:
    if row["CosSimsS4E1"] > row["CosSimsS4E2"]:
      correctCount = correctCount + 1
    elif row["CosSimsS4E2"] > row["CosSimsS4E1"]:
      incorrectCount = incorrectCount + 1
  elif row["CorrectAnswers"] == 2:
    if row["CosSimsS4E2"] > row["CosSimsS4E1"]:
      correctCount = correctCount + 1
    elif row["CosSimsS4E1"] > row["CosSimsS4E2"]:
      incorrectCount = incorrectCount + 1


In [97]:
print("S3BERT Model Sentence by Sentence Cosine Similaritites, Correctly Identifies Matching Narrative Ending\n - Correct Narrative Ending Selected (n = {} times)\n - Incorrect Narrative Ending Selected (n = {} times).\nS3BERT Model Accuracy: {}".format(correctCount, incorrectCount, correctCount / (incorrectCount + correctCount)))

S3BERT Model Sentence by Sentence Cosine Similaritites, Correctly Identifies Matching Narrative Ending
 - Correct Narrative Ending Selected (n = 1013 times)
 - Incorrect Narrative Ending Selected (n = 558 times).
S3BERT Model Accuracy: 0.6448122215149587


References

Opitz, J. & Frank, A. (2022). “SBERT studies meaning representations: Decomposing sentence embeddings into explainable semantic features.” In He, Y., Ji, H., Li, S., Liu, Y., and Chang, C.-H., editors, Proceedings of the 2nd Conference of the Asia-Pacific Chapter of the Association for Computational Linguistics and the 12th International Joint Conference on Natural Language Processing (Volume 1: Long Papers), pages 625–638, Online only. Association for Computational Linguistics. https://aclanthology.org/2022.aacl-main.48.pdf